HomeGuard Security System Simulator

Author: Marc Tanguy Durán

Description: A smart home monitoring system that processes sensor readings
             and triggers alerts for security, safety, and comfort issues.

#### **Pre-work**

##### The first part of the lab sets up the functions and initial testing necessary for the final simulation success.

In [ ]:
import random
from datetime import datetime
 
# System configuration
HOME_MODES = ["HOME", "AWAY", "SLEEP"]
ALERT_SEVERITIES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
 
# Current system state
current_mode = "AWAY"

Here we imported the necessary libraries and defined the different system modes and alert severities that will be used throughout the initial testing and final simulation run. For the testing, we set the mode as "Away".

In [5]:
def create_sensor(sensor_id, location, sensor_type, threshold=None):
    """
    Creates a sensor data structure.
    
    Parameters:
    - sensor_id: Unique identifier for the sensor
    - location: Where the sensor is located (e.g., "Living Room")
    - sensor_type: Type of sensor ("motion", "temperature", "door", "smoke")
    - threshold: Optional threshold value for the sensor
    
    Returns:
    - A dictionary representing the sensor
    """
    sensor = {
        "id": sensor_id,
        "location": location,
        "type": sensor_type,
        "threshold": threshold,
        "current_threshold_value": None
    }
    return sensor
 
def create_alert(severity, message, sensor_id, timestamp):
    """
    Creates an alert data structure.
    
    Parameters:
    - severity: Alert severity level (LOW, MEDIUM, HIGH, CRITICAL)
    - message: Description of the alert
    - sensor_id: ID of the sensor that triggered the alert
    - timestamp: When the alert was triggered
    
    Returns:
    - A dictionary representing the alert
    """
    alert = {
        "severity": severity,
        "message": message,
        "sensor_id": sensor_id,
        "timestamp": timestamp 
    }
    return alert
 
# Initialize sensors for the Peterson home
sensors = [
    create_sensor("MOTION_001", "Living Room", "motion"),
    create_sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
    create_sensor("DOOR_001", "Front Door", "door"),
    create_sensor("SMOKE_001", "Bedroom", "smoke")
]

In [6]:
print(f"Initialized {len(sensors)} sensors")
for sensor in sensors:
    print(f"  - {sensor['id']}: {sensor['location']} ({sensor['type']})")

Initialized 4 sensors
  - MOTION_001: Living Room (motion)
  - TEMP_001: Kitchen (temperature)
  - DOOR_001: Front Door (door)
  - SMOKE_001: Bedroom (smoke)


The above cells defined the sensors and alert creation functions and tested the initialisation of our sensors above defined list.

In [7]:
def is_abnormal_reading(sensor, reading_value):
    """
    Checks if a sensor reading is abnormal based on sensor type and thresholds.
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The current reading from the sensor
    
    Returns:
    - True if the reading is abnormal, False otherwise
    """
    sensor_type = sensor["type"]
    
    if sensor_type == "temperature":
        if reading_value < 35 or reading_value > 95:
            return True
        else:
            return False
    
    elif sensor_type == "motion":
        if reading_value == True:
            return True
        else:
            return False
    
    elif sensor_type == "door":
        if reading_value == "OPEN":
            return True
        else:
            return False
    
    elif sensor_type == "smoke":
        if reading_value == "DETECTED":
            return True
        else:
            return False
    
 
 
def should_trigger_security_alert(sensor, reading_value, system_mode):
    """
    Determines if a security alert should be triggered.
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The current reading from the sensor
    - system_mode: Current system mode (HOME, AWAY, SLEEP)
    
    Returns:
    - True if a security alert should be triggered, False otherwise
    """
    if system_mode == "AWAY":
        if sensor["type"] == "motion" and reading_value == True:
            return True
        elif sensor["type"] == "door" and reading_value == "OPEN":
            return True
    return False

In [8]:
# Test temperature check
test_sensor = create_sensor("TEMP_TEST", "Test Room", "temperature", threshold=35)
print(f"34°F is abnormal: {is_abnormal_reading(test_sensor, 34)}")  # Should be True
print(f"68°F is abnormal: {is_abnormal_reading(test_sensor, 68)}")  # Should be False
 
# Test security alert
motion_sensor = create_sensor("MOTION_TEST", "Test Room", "motion")
print(f"Motion in AWAY mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'AWAY')}")  # Should be True
print(f"Motion in HOME mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'HOME')}")  # Should be False

34°F is abnormal: True
68°F is abnormal: False
Motion in AWAY mode triggers alert: True
Motion in HOME mode triggers alert: False


The above cells defined and tested the functions for when the system should consider a value reading abnormal and when it should trigger a security alert.

In [ ]:
def generate_reading(sensor):
    """
    Generates a realistic reading for a sensor.
    
    Parameters:
    - sensor: Sensor dictionary
    
    Returns:
    - A realistic reading value based on sensor type
    """    
    sensor_type = sensor["type"]
    
    if sensor_type == "temperature":
        return random.randint(30, 100)
    elif sensor_type == "motion":
        return random.choice([True, False])
    elif sensor_type == "door":
        return random.choice(["OPEN", "CLOSED"])
    elif sensor_type == "smoke":
        return random.choice(["CLEAR", "DETECTED"])
 
 
def process_reading(sensor, reading_value, system_mode):
    """
    Processes a sensor reading and determines if alerts are needed.
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The reading from the sensor
    - system_mode: Current system mode
    
    Returns:
    - A list of alerts (empty if no alerts needed)
    """
    alerts = []
    timestamp = datetime.now().strftime("%H:%M:%S")
    
     # Security alerts
    if should_trigger_security_alert(sensor, reading_value, system_mode):
        if sensor["type"] == "motion":
            alerts.append(create_alert("HIGH", f"[SECURITY] Motion detected in {sensor['location']} while AWAY", sensor["id"], timestamp))
        elif sensor["type"] == "door":
            alerts.append(create_alert("HIGH", f"[SECURITY] {sensor['location']} opened while AWAY", sensor["id"], timestamp))
    
    # Safety alerts
    if sensor["type"] == "temperature":
        if reading_value < 35:
            alerts.append(create_alert("HIGH", f"[SAFETY] Frozen pipe risk: Temperature is {reading_value}°F in {sensor['location']}", sensor["id"], timestamp))
        elif reading_value > 95:
            alerts.append(create_alert("HIGH", f"[SAFETY] Equipment failure risk: Temperature is {reading_value}°F in {sensor['location']}", sensor["id"], timestamp))
    
    if sensor["type"] == "smoke" and reading_value == "DETECTED":
        alerts.append(create_alert("CRITICAL", f"[SAFETY] Smoke detected in {sensor['location']}. Fire risk!", sensor["id"], timestamp))
    
    # Comfort notifications (only in HOME mode)
    if system_mode == "HOME" and sensor["type"] == "temperature":
        if reading_value < 65 or reading_value > 75:
            alerts.append(create_alert("LOW", f"[COMFORT] Temperature is {reading_value}°F in {sensor['location']}, outside comfort range.", sensor["id"], timestamp))
    
    return alerts
 
# Feel free to change the function signatures or add helper functions as needed, these are just some suggestions to make the code more friendly! 
def trigger_alert(alert):
    """
    Displays an alert to the user.
    
    Parameters:
    - alert: Alert dictionary
    """
    severity_symbol = {
        "LOW": "ℹ️",
        "MEDIUM": "⚠️",
        "HIGH": "🚨",
        "CRITICAL": "🔥"
    }
    
    symbol = severity_symbol.get(alert["severity"], "⚠️")
    print(f"[ALERT!] {symbol} {alert['severity']}: {alert['message']}")
 
def log_event(message, timestamp=None):
    """
    Logs an event to the console.
    
    Parameters:
    - message: The message to log
    - timestamp: Optional timestamp (uses current time if not provided)
    """
    if timestamp is None:
        timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")

In [15]:
# Test reading generation
test_sensor = sensors[0]  # Motion sensor
reading = generate_reading(test_sensor)
print(f"Generated reading for {test_sensor['location']}: {reading}")
 
# Test processing
alerts = process_reading(test_sensor, True, "AWAY")
if alerts:
    trigger_alert(alerts[0])

Generated reading for Living Room: False
[ALERT!] 🚨 HIGH: [SECURITY] Motion detected in Living Room while AWAY


The above cells defined and tested the functions of generating a reading and processing it. In the process reading function, we generated the conditions for an alert addition to the empty alert list based on said reading interpretation. Then, we defined the function of triggering those alerts and logging events events.

In [16]:
class Sensor:
    """
    Represents a sensor in the HomeGuard system.
    """
    
    def __init__(self, sensor_id, location, sensor_type, threshold=None):
        """
        Initializes a new sensor.
        
        Parameters:
        - sensor_id: Unique identifier for the sensor
        - location: Where the sensor is located
        - sensor_type: Type of sensor ("motion", "temperature", "door", "smoke")
        - threshold: Optional threshold value for the sensor
        """
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.threshold = threshold
        self.current_value = None
 
    
    def read(self):
        """
        Generates and stores a new reading for this sensor.
        
        Returns:
        - The reading value
        """
        if self.type == "temperature":
            self.current_value = random.randint(30, 100)
        elif self.type == "motion":
            self.current_value = random.choice([True, False])
        elif self.type == "door":
            self.current_value = random.choice(["OPEN", "CLOSED"])
        elif self.type == "smoke":
            self.current_value = random.choice(["CLEAR", "DETECTED"])
        return self.current_value
        
    def isAbnormal(self):
        """
        Checks if the current reading is abnormal.
        
        Returns:
        - True if the reading is abnormal, False otherwise
        """
        if self.type == "temperature":
            if self.current_value < 35 or self.current_value > 95:
                return True
            else:
                return False

        elif self.type == "motion":
            if self.current_value == True:
                return True
            else:
                return False
    
        elif self.type == "door":
            if self.current_value == "OPEN":
                return True
            else:
                return False
    
        elif self.type == "smoke":
            if self.current_value == "DETECTED":
                return True
            else:
                return False
  
    
    def reset(self):
        """
        Resets the sensor's current reading to None.
        """
        self.current_value = None
    
    def __str__(self):
        """
        Returns a string representation of the sensor.
        """
        status = "No reading" if self.current_value is None else str(self.current_value)
        return f"{self.id} ({self.location}): {status}"
 
# Create sensor objects using the class
sensor_objects = [
    Sensor("MOTION_001", "Living Room", "motion"),
    Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
    Sensor("DOOR_001", "Front Door", "door"),
    Sensor("SMOKE_001", "Bedroom", "smoke")
]

In [17]:
test_sensor = Sensor("TEST_001", "Test Room", "temperature", threshold=35)
test_sensor.read()
print(f"Sensor reading: {test_sensor.current_value}")
print(f"Is abnormal: {test_sensor.isAbnormal()}")
print(f"Sensor info: {test_sensor}")

Sensor reading: 83
Is abnormal: False
Sensor info: TEST_001 (Test Room): 83


The above cells combined the data and the functions into a single unit (Class) basically defining how the properties integrate with the methods system works and tested it.

#### **Simulation**

##### The second part of the lab establishes how the system works based on the above defined functions and simulates runs.

In [23]:
def run_simulation(duration_minutes=5, system_mode="AWAY"):
    """
    Runs the HomeGuard security system simulation.
    
    Parameters:
    - duration_minutes: How long to run the simulation (default: 5 minutes)
    - system_mode: System mode (HOME, AWAY, SLEEP)
    """
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")
    
    # Use sensor objects instead of dictionaries
    sensors = [
        Sensor("MOTION_001", "Living Room", "motion"),
        Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
        Sensor("DOOR_001", "Front Door", "door"),
        Sensor("SMOKE_001", "Bedroom", "smoke")
    ]
    
    # Simulate time passing (each iteration = 1 minute)
    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")

        motion_triggered = False
        door_triggered = False
        
        # Read all sensors
        for sensor in sensors:
            reading = sensor.read()
            
            # Display the reading
            if sensor.type == "temperature":
                status = "Normal" if 65 <= reading <= 75 else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")
            elif sensor.type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")
            
            if sensor.type == "motion" and reading == True:
                motion_triggered = True
            if sensor.type == "door" and reading == "OPEN":
                door_triggered = True
    
            # Process the reading and trigger alerts if needed
            # Convert sensor object to dict format for process_reading function
            sensor_dict = {
                "id": sensor.id,
                "location": sensor.location,
                "type": sensor.type,
                "threshold": sensor.threshold
            }
            alerts = process_reading(sensor_dict, reading, system_mode)
            
            # Trigger all alerts
            for alert in alerts:
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")
        
        if motion_triggered and door_triggered and system_mode == "AWAY":
            timestamp = datetime.now().strftime("%H:%M:%S")
            alert = create_alert("CRITICAL", "Possible break-in! Motion detected and door opened!", "MULTIPLE", timestamp)
            trigger_alert(alert)
            log_event("Sending notification to homeowner...")
        
        # Small delay to make output readable (optional)
        import time
        time.sleep(0.5)  # 0.5 second delay between iterations
    
    print("\n" + "=" * 50)
    print("Simulation complete!")
    print("=" * 50)
 
# Main execution
if __name__ == "__main__":
    # Run the simulation
    run_simulation(duration_minutes=3, system_mode="AWAY")

=== HomeGuard Security System ===
Mode: AWAY


Time: 18:54:01
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 71°F (Normal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 18:54:01
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: [SECURITY] Motion detected in Living Room while AWAY
[LOG] [18:54:01] Sending notification to homeowner...
[READING] Kitchen Temperature: 59°F (Abnormal)
[READING] Front Door: OPEN
[ALERT!] 🚨 HIGH: [SECURITY] Front Door opened while AWAY
[LOG] [18:54:01] Sending notification to homeowner...
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: [SAFETY] Smoke detected in Bedroom. Fire risk!
[LOG] [18:54:01] Sending notification to homeowner...
[ALERT!] 🔥 CRITICAL: Possible break-in! Motion detected and door opened!
[LOG] [18:54:01] Sending notification to homeowner...

Time: 18:54:02
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 76°F (Abnormal)
[READING] Front Door: CLOSED
[READING] 

The above cell brings everything together: reads all sensors, processes alerts and logs events the way a live security system would do through the random generation of values. 

#### Step 7: Testing the system

##### **Security test:**

In [24]:
run_simulation(duration_minutes=3, system_mode="AWAY")

=== HomeGuard Security System ===
Mode: AWAY


Time: 18:54:08
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 41°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: [SAFETY] Smoke detected in Bedroom. Fire risk!
[LOG] [18:54:08] Sending notification to homeowner...

Time: 18:54:09
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 52°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: [SAFETY] Smoke detected in Bedroom. Fire risk!
[LOG] [18:54:09] Sending notification to homeowner...

Time: 18:54:09
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: [SECURITY] Motion detected in Living Room while AWAY
[LOG] [18:54:09] Sending notification to homeowner...
[READING] Kitchen Temperature: 91°F (Abnormal)
[READING] Front Door: OPEN
[ALERT!] 🚨 HIGH: [SECURITY] Front Door opened while AWAY
[LOG] [18:54:09] Sending notification to homeowner...
[READING] 

Here we run the simulation in AWAY mode to verify motion and door alerts trigger correctly.

##### **Safety test:**

In [30]:
run_simulation(duration_minutes=3, system_mode="SLEEP")

=== HomeGuard Security System ===
Mode: SLEEP


Time: 19:01:06
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 32°F (Abnormal)
[ALERT!] ⚠️ MEDIUM: [SAFETY] Frozen pipe risk: Temperature is 32°F in Kitchen
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: [SAFETY] Smoke detected in Bedroom. Fire risk!
[LOG] [19:01:06] Sending notification to homeowner...

Time: 19:01:07
[READING] Living Room Motion: DETECTED
[READING] Kitchen Temperature: 59°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 19:01:07
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 97°F (Abnormal)
[ALERT!] ⚠️ MEDIUM: [SAFETY] Equipment failure risk: Temperature is 97°F in Kitchen
[READING] Front Door: OPEN
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: [SAFETY] Smoke detected in Bedroom. Fire risk!
[LOG] [19:01:07] Sending notification to homeowner...

Simulation complete!


Here we run the simulation in SLEEP mode to verify that smoke and temperature alerts trigger regardless of system mode.

##### **Comfort check:**

In [26]:
run_simulation(duration_minutes=3, system_mode="HOME")

=== HomeGuard Security System ===
Mode: HOME


Time: 18:56:44
[READING] Living Room Motion: DETECTED
[READING] Kitchen Temperature: 33°F (Abnormal)
[ALERT!] ⚠️ MEDIUM: [SAFETY] Frozen pipe risk: Temperature is 33°F in Kitchen
[ALERT!] ℹ️ LOW: [COMFORT] Temperature is 33°F in Kitchen, outside comfort range.
[READING] Front Door: OPEN
[READING] Bedroom Smoke: CLEAR

Time: 18:56:44
[READING] Living Room Motion: DETECTED
[READING] Kitchen Temperature: 61°F (Abnormal)
[ALERT!] ℹ️ LOW: [COMFORT] Temperature is 61°F in Kitchen, outside comfort range.
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 18:56:45
[READING] Living Room Motion: DETECTED
[READING] Kitchen Temperature: 55°F (Abnormal)
[ALERT!] ℹ️ LOW: [COMFORT] Temperature is 55°F in Kitchen, outside comfort range.
[READING] Front Door: OPEN
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: [SAFETY] Smoke detected in Bedroom. Fire risk!
[LOG] [18:56:45] Sending notification to homeowner...

Simulation complete!


Here we run the simulation in HOME mode to verify that comfort alerts trigger correctly.